# Update ROR Lookup data by adding lang attribute to organization names
This is a capability of ROR Schema Version 2 that needs to be added to existing RORRetriever output files

In [ ]:
import pandas as pd
import requests

In [ ]:
dataDirectory = '~//MetadataGameChanger/Repositories/repositoryReCuration/data/'
ror_lookup_df = pd.read_excel(dataDirectory + "RORLookup.xlsx")
ror_lookup_df.head()

In [ ]:
ror_lookup_df['lang'] = None
for i, ind in enumerate(ror_lookup_df.index):
    ror = ror_lookup_df.loc[ind,'ROR_Affiliation']
    if not isinstance(ror,str) or (ror is None) or (ror == ''):
        continue
    
    ror = ror.split('/')[-1]
    URL = 'https://api.ror.org/v2/organizations/' + ror
    response = requests.get(URL)
    if response.status_code == 200:
        item = response.json()
        name = [ x['value'] for x in item['names'] if 'ror_display' in x['types'] ][0]
        lang = [ x['lang']  for x in item['names'] if 'ror_display' in x['types'] ][0]
        ror_lookup_df.at[ind,'organizationLookupName_Affiliation'] = name
        ror_lookup_df.at[i,'lang'] = lang
        print(f"{i} {ror}: {name} ({lang})")

In [ ]:
ror_lookup_df['lang'] = ror_lookup_df['lang'].fillna('en')

In [ ]:
ror_lookup_df['lang'].value_counts()

In [ ]:
ror_lookup_df.to_csv(dataDirectory + "RORLookup_withLang.csv", index=False)